# AutoFeatureFE — Single-Dataset Demo

An LLM agent iteratively proposes feature-engineering pipelines;
a fixed XGBoost model evaluates them. Only the features change.

**Works with**: Titanic (CSV), sklearn built-ins, or any OpenML dataset.
Change Section 2 to swap datasets with no other code changes needed.

In [ ]:
# Install dependencies (skip if already installed)
%pip install -q xgboost scikit-learn pandas numpy matplotlib seaborn anthropic openai openml

## 1. Setup

In [ ]:
import sys, json, shutil, importlib
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

REPO_DIR = Path('.').resolve()
sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

## 2. Choose a dataset

Pick **one** of the blocks below (comment the others out).

In [ ]:
# ── Option A: Titanic CSV (classic binary classification) ─────────────────────
import seaborn as sns
raw = sns.load_dataset('titanic')
target_col = 'survived'

# Drop high-cardinality string cols the agent can't use; keep the rest for auto_encode
drop_cols = ['name', 'ticket', 'cabin', 'boat', 'body', 'home.dest', 'embarked_town']
df_save = raw.drop(columns=[c for c in drop_cols if c in raw.columns])
df_save.to_csv(REPO_DIR / 'titanic.csv', index=False)

TASK_CFG = {
    "task": "classification", "dataset": "csv",
    "csv_path": "titanic.csv", "target_column": target_col, "metric": "auc"
}
print(f"Saved titanic.csv  ({df_save.shape[0]} rows, {df_save.shape[1]-1} features)")
df_save.head(3)

In [ ]:
# ── Option B: California Housing (sklearn, regression) ────────────────────────
# TASK_CFG = {"task": "regression", "dataset": "california_housing", "metric": "rmse"}

# ── Option C: Breast Cancer (sklearn, binary classification) ──────────────────
# TASK_CFG = {"task": "classification", "dataset": "breast_cancer", "metric": "auc"}

# ── Option D: Any OpenML dataset by ID ────────────────────────────────────────
# TASK_CFG = {"task": "classification", "dataset": "openml", "openml_id": 31, "metric": "auc"}

# ── Option E: OpenML benchmark suite dataset by index ─────────────────────────
# TASK_CFG = {"task": "regression", "dataset": "openml",
#             "openml_suite": "OpenML-CTR23", "openml_suite_index": 0, "metric": "rmse"}

## 3. Configure & inspect

In [ ]:
# Write task.json and reset pipeline to baseline
(REPO_DIR / 'task.json').write_text(json.dumps(TASK_CFG, indent=2))
(REPO_DIR / 'pipeline.json').write_text(json.dumps({
    "description": "baseline — standard scaling only",
    "steps": [{"op": "scale", "method": "standard"}]
}, indent=2))
for f in ['results.tsv', 'topk_pool.json']:
    p = REPO_DIR / f
    if p.exists(): p.unlink()

print(json.dumps(TASK_CFG, indent=2))

In [ ]:
# Inspect field types (auto-detected before pipeline)
import importlib
import prepare as _prepare; importlib.reload(_prepare)
import field_types as _ft;  importlib.reload(_ft)

raw_field_types = _prepare.get_field_types()
feature_names   = _prepare.get_feature_names()

ft_df = pd.DataFrame([
    {"feature": col, "inferred_type": t}
    for col, t in raw_field_types.items()
])
print(f"Features after auto-encoding: {feature_names}")
print()
ft_df

In [ ]:
# Preview data after auto-encoding (what the pipeline receives)
from field_types import auto_encode

task_cfg = json.loads((REPO_DIR / 'task.json').read_text())
df_raw, y, openml_feats = _prepare._load_raw(task_cfg)

dummy = df_raw.head(5).reset_index(drop=True)
enc_tr, enc_va, report = auto_encode(dummy.copy(), dummy.copy(), raw_field_types)

print("Auto-encoding report:")
if report['dropped_id']:        print(f"  Dropped (ID cols): {report['dropped_id']}")
if report['expanded_datetime']: print(f"  Expanded (datetime): {report['expanded_datetime']}")
if report['encoded_categorical']: print(f"  Ordinal-encoded: {report['encoded_categorical']}")

enc_tr

## 4. Baseline evaluation

In [ ]:
importlib.reload(_prepare)
X_train, X_val, y_train, y_val = _prepare.prepare_data()
print(f"Train: {X_train.shape}  Val: {X_val.shape}")
print(f"y: dtype={y_train.dtype}  classes={np.unique(y_train) if TASK_CFG['task']=='classification' else 'continuous'}")

In [ ]:
from xgboost import XGBClassifier, XGBRegressor
from sklearn.metrics import roc_auc_score

_XGB_BASE = dict(n_estimators=1000, max_depth=6, learning_rate=0.05,
                 subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
                 reg_lambda=1.0, random_state=42, n_jobs=-1,
                 early_stopping_rounds=50, verbosity=0)

task_type = TASK_CFG["task"]
metric    = TASK_CFG["metric"]

if task_type == "regression":
    baseline_model = XGBRegressor(**_XGB_BASE, eval_metric="rmse")
    baseline_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    from sklearn.metrics import root_mean_squared_error
    baseline_score = root_mean_squared_error(y_val, baseline_model.predict(X_val))
    print(f"Baseline RMSE: {baseline_score:.4f}")
else:
    n_classes = len(np.unique(y_train))
    obj = "binary:logistic" if n_classes == 2 else "multi:softprob"
    extra = {} if n_classes == 2 else {"num_class": n_classes}
    baseline_model = XGBClassifier(**_XGB_BASE, objective=obj, eval_metric="auc" if n_classes==2 else "mlogloss", **extra)
    baseline_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    proba = baseline_model.predict_proba(X_val)
    baseline_score = roc_auc_score(y_val, proba[:,1] if n_classes==2 else proba, multi_class="ovr", average="macro")
    print(f"Baseline AUC: {baseline_score:.4f}")

## 5. Run the agent loop

Paste your API key and set `n_iterations`. Resume any time by re-running this cell.

In [ ]:
import agent as _agent; importlib.reload(_agent)

_agent.run(
    n_iterations         = 10,
    anthropic_api_key    = "sk-ant-...",   # ← paste your key (or use openai_api_key)
    # openai_api_key     = "sk-...",
    topk                 = 5,
    complexity_alpha     = 0.005,
    include_history      = True,
    history_size         = 20,
    include_data_profile = True,
    allow_freestyle      = False,
    working_dir          = str(REPO_DIR),
)

## 6. Results

In [ ]:
results_path = REPO_DIR / 'results.tsv'
if not results_path.exists():
    print("No results yet — run Section 5 first.")
else:
    results = pd.read_csv(results_path, sep='\t')
    metric_name = results['metric_name'].iloc[0] if 'metric_name' in results.columns else metric
    higher_is_better = metric_name == 'auc'
    running_best = results['val_score'].cummax() if higher_is_better else results['val_score'].cummin()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Score over iterations
    axes[0].scatter(results.index, results['val_score'], alpha=0.6, zorder=3, label='Each run')
    axes[0].plot(running_best, color='crimson', lw=2, label='Running best')
    axes[0].axhline(baseline_score, ls='--', color='grey', label=f'Baseline ({baseline_score:.4f})')
    axes[0].set(xlabel='Iteration', ylabel=metric_name.upper(), title='Score per iteration')
    axes[0].legend()

    # Feature count
    if 'n_features' in results.columns:
        axes[1].bar(results.index, results['n_features'], color='steelblue', alpha=0.7)
        axes[1].set(xlabel='Iteration', ylabel='# Features', title='Feature count per iteration')

    plt.suptitle(f'AutoFeatureFE — {TASK_CFG.get("dataset","")} [{metric_name.upper()}]',
                 fontsize=13, fontweight='bold')
    plt.tight_layout(); plt.show()

    best_idx   = results['val_score'].idxmax() if higher_is_better else results['val_score'].idxmin()
    best_score = results.loc[best_idx, 'val_score']
    delta      = (best_score - baseline_score) / abs(baseline_score) * 100
    sign       = '+' if higher_is_better else ''
    print(f"Baseline : {baseline_score:.4f}")
    print(f"Best     : {best_score:.4f}  (iter {best_idx})  Δ = {sign}{delta:.2f}%")

## 7. Best pipeline

In [ ]:
pool_path = REPO_DIR / 'topk_pool.json'
if pool_path.exists():
    pool = json.loads(pool_path.read_text())
    for i, e in enumerate(pool, 1):
        print(f"Rank {i}: score={e['score']:.6f}  '{e['description']}'")
        print(f"  ops: {[s['op'] for s in e['steps']]}\n")

print("Current pipeline.json:")
print(json.dumps(json.loads((REPO_DIR/'pipeline.json').read_text())['steps'], indent=2))

In [ ]:
# Feature importance from best pipeline
importlib.reload(_prepare)
X_tr_b, X_va_b, y_tr_b, y_va_b = _prepare.prepare_data()

if task_type == "regression":
    best_model = XGBRegressor(**_XGB_BASE, eval_metric="rmse")
    best_model.fit(X_tr_b, y_tr_b, eval_set=[(X_va_b, y_va_b)], verbose=False)
else:
    best_model = XGBClassifier(**_XGB_BASE, objective=obj, eval_metric="auc" if n_classes==2 else "mlogloss", **extra)
    best_model.fit(X_tr_b, y_tr_b, eval_set=[(X_va_b, y_va_b)], verbose=False)

fn = _prepare.get_feature_names() + [f"f{i}" for i in range(X_tr_b.shape[1])]
fn = fn[:X_tr_b.shape[1]]
fi_df = pd.DataFrame({'feature': fn, 'importance': best_model.feature_importances_})
fi_df = fi_df.nlargest(20, 'importance').sort_values('importance')

fi_df.plot.barh(x='feature', y='importance', figsize=(8, 6), legend=False, color='steelblue')
plt.title('Top-20 Feature Importances (best pipeline)')
plt.tight_layout(); plt.show()

## Next steps

| What | How |
|---|---|
| Run longer | `run(n_iterations=50, ...)` — resumes automatically if pool exists |
| Different dataset | Change `TASK_CFG` in Section 2, re-run from there |
| OpenML dataset | `TASK_CFG = {"dataset":"openml","openml_id":31,...}` |
| Allow creative ops | `allow_freestyle=True` in `run()` |
| View history | `column -t -s $'\t' results.tsv` in terminal |
| Run many datasets | See **benchmark_openml.ipynb** |